# Silver: PDFs to Markdown

One table, `silver_pdfs`, filled in separate cells:

```
bronze_pdfs --ai_parse_document--> silver_pdfs.parsed --ai_query--> silver_pdfs.markdown
```

Incremental: step 2 streams from bronze with a checkpoint, and step 3 only converts rows with no markdown yet.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("endpoint", "databricks-qwen3-next-80b-a3b-instruct")
dbutils.widgets.text("max_files_per_batch", "10")  # step 2: bronze files per micro-batch
dbutils.widgets.text("pages_per_call", "3")        # step 3: pages sent to ai_query per call
dbutils.widgets.text("docs_per_batch", "50")       # step 3: documents per batch (save point)

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
endpoint = dbutils.widgets.get("endpoint")
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))
pages_per_call = int(dbutils.widgets.get("pages_per_call"))
docs_per_batch = int(dbutils.widgets.get("docs_per_batch"))

bronze = f"{catalog}.{schema}.bronze_pdfs"
silver = f"{catalog}.{schema}.silver_pdfs"
checkpoint = f"/Volumes/{catalog}/{schema}/docs/_checkpoints/silver_pdfs"

## 1. Create the table

`parts` holds the Markdown of each converted part (part number to text) and `num_parts` how many parts the document has. They let step 3 resume where it stopped.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver} (
  path STRING,
  parsed VARIANT,
  markdown STRING,
  parts MAP<INT, STRING>,
  num_parts INT
)
""")

# Table created by an earlier version: add the two new columns.
if "parts" not in spark.table(silver).columns:
    spark.sql(f"ALTER TABLE {silver} ADD COLUMNS (parts MAP<INT, STRING>, num_parts INT)")

## 2. Parse with `ai_parse_document` (streaming)

Reads `bronze_pdfs` as a stream and appends the parsed output to `silver_pdfs`. `maxFilesPerTrigger` keeps each micro-batch small, so progress is committed often and a failure only loses the current batch. The checkpoint remembers which bronze rows were already processed, so each run parses only new PDFs. `trigger(availableNow=True)` processes everything new and stops. `markdown` stays empty until step 3.

In [ ]:
(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).table(bronze)
    .select(
        "path",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
        F.lit(None).cast("string").alias("markdown"),
        F.lit(None).cast("map<int,string>").alias("parts"),
        F.lit(None).cast("int").alias("num_parts"),
    )
    .writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(silver)
    .awaitTermination()
)
print(spark.table(silver).count(), "parsed")

## 3. Convert to Markdown with `ai_query`

One run converts every document that has no `markdown` yet. Each document is split into parts of `pages_per_call` pages, so every call's output stays under the endpoint's 10000-token cap, and Spark runs the parts in parallel. The cell works through the documents in batches of `docs_per_batch` and saves each batch before starting the next:

1. Only parts not already in `parts` are sent to `ai_query`.
2. Successful parts are merged into `parts`. Failed parts are simply not stored.
3. `markdown` is built for documents that now have all `num_parts` parts.

A failure or rerun only repeats the parts that are still missing. The loop stops if a batch stores no new part.

In [ ]:
PROMPT = """Convert these parsed elements of a PDF (JSON from ai_parse_document) into Markdown.

Each element has "page_no", "type" and "content", in reading order.

Rules:
- Keep ALL content exactly as written. Do not summarize, reword or invent anything.
- Titles become "# Title"; section headers become "##" / "###".
- Keep plain text as paragraphs and lists as Markdown lists.
- Convert tables (given as HTML) to Markdown tables (header row, | --- | separator row), keeping every row and column.
- Keep metadata such as page headers, footers and page numbers as italic lines.
- Start each page with the line "<!-- page N -->" (N = page_no). Between two pages put a blank line, "---", and a blank line.
- Output ONLY the Markdown, with no code fences or commentary.

Elements:
"""

from pyspark.sql import Window

def convert_batch():
    """Convert up to docs_per_batch documents. Returns the number of failed parts."""
    pending_docs = spark.table(silver).filter("markdown IS NULL").limit(docs_per_batch)
    pending_docs.createOrReplaceTempView("pending_docs")

    # One row per element, with its page number counted 1, 2, 3 per document.
    els = spark.sql("""
    SELECT p.path, e.pos,
           coalesce(e.value:bbox[0].page_id::INT, 0) AS page_id,
           e.value:type::STRING AS type,
           e.value:content::STRING AS content
    FROM pending_docs p, LATERAL variant_explode(p.parsed:document:elements) e
    """).withColumn("page_no", F.dense_rank().over(Window.partitionBy("path").orderBy("page_id")))

    # Group pages into parts, one ai_query call per part.
    parts = (
        els.withColumn("part", F.floor((F.col("page_no") - 1) / pages_per_call).cast("int"))
        .groupBy("path", "part")
        .agg(
            F.max("page_no").alias("last_page"),
            F.array_sort(F.collect_list(F.struct("pos", "page_no", "type", "content"))).alias("els"),
        )
        .withColumn("num_parts", F.ceil(F.max("last_page").over(Window.partitionBy("path")) / pages_per_call).cast("int"))
        .withColumn("input", F.to_json(F.transform("els", lambda e: F.struct(e["page_no"], e["type"], e["content"]))))
        .withColumn("prompt", F.concat(F.lit(PROMPT), F.col("input")))
    )

    # Skip parts already stored for the document.
    todo = (
        parts.join(pending_docs.select("path", F.col("parts").alias("done")), "path")
        .filter("done IS NULL OR NOT map_contains_key(done, part)")
    )

    converted = todo.repartition(32).withColumn("r", F.expr(
        f"ai_query('{endpoint}', prompt, "
        "modelParameters => named_struct('temperature', 0.0, 'max_tokens', 10000), "
        "failOnError => false)"
    )).cache()

    failed = converted.filter("r.errorMessage IS NOT NULL OR r.result IS NULL").count()
    print(f"parts failed (retried on next run): {failed}")

    # Successful parts per document.
    new_parts = (
        converted.filter("r.errorMessage IS NULL AND r.result IS NOT NULL")
        .groupBy("path", "num_parts")
        .agg(F.map_from_entries(F.collect_list(F.struct("part", F.col("r.result")))).alias("new_parts"))
    )

    # Add them to the stored parts. When all parts exist, build the markdown in part order.
    merged = "map_concat(coalesce(t.parts, cast(map() AS MAP<INT, STRING>)), s.new_parts)"
    (
        DeltaTable.forName(spark, silver).alias("t")
        .merge(new_parts.alias("s"), "t.path = s.path")
        .whenMatchedUpdate(set={
            "parts": merged,
            "num_parts": "s.num_parts",
            "markdown": f"CASE WHEN size({merged}) = s.num_parts THEN "
                        f"concat_ws('\\n\\n---\\n\\n', transform(array_sort(map_entries({merged})), x -> x.value)) END",
        })
        .execute()
    )
    converted.unpersist()
    return failed


def stored_parts():
    return spark.table(silver).selectExpr("coalesce(sum(size(parts)), 0)").first()[0]

def docs_left():
    return spark.table(silver).filter("markdown IS NULL").count()

# One run converts every document. Each batch is merged (saved) before the next one starts.
batch = 0
while docs_left() > 0:
    batch += 1
    before = stored_parts()
    failed = convert_batch()
    print(f"batch {batch}: {stored_parts() - before} parts stored, {failed} failed, {docs_left()} documents left")
    if stored_parts() == before:
        print("No progress in this batch, stopping. Check the failed parts, or lower pages_per_call.")
        break

## 4. Check

In [ ]:
r = spark.table(silver).filter("markdown IS NOT NULL").limit(1).collect()
if r:
    print(r[0]["path"]); print(r[0]["markdown"][:2000])

## Notes

- Full rebuild: drop `silver_pdfs`, delete the checkpoint folder (`/Volumes/<catalog>/<schema>/docs/_checkpoints/silver_pdfs`) and rerun. Dropping only the table leaves the checkpoint, so nothing would be reparsed.
- To re-run only the prompt: `UPDATE silver_pdfs SET markdown = NULL, parts = NULL, num_parts = NULL`, then rerun step 3.
- `docs_per_batch` is the save point, not a limit on the run: the loop keeps going until no document is left. A failure loses at most the current batch, and a rerun only redoes missing parts. If a whole batch stores nothing the loop stops. Check the failed-part count and lower `pages_per_call` if parts keep failing.
- Set the `endpoint` parameter to a Foundation Model endpoint in your workspace. This endpoint caps output at 10000 tokens.
- `ai_parse_document` and `ai_query` are billed per use. Test on a few PDFs first.